In [0]:
%python
"""
Test extract_county (and the project/org coalesce) against the labelled
missing_program_county spreadsheet.

Usage:
    python tests/test_extract_county.py path/to/missing_program_county_sql_338.xlsx

Expected values come from the spreadsheet:
    project_name_county_match -> extract_county("project_name")
    org_name_county_match     -> extract_county("organization_name")
    lookup_program_county     -> coalesce(project, org); NULL for 'unresolved' rows
The string "null" in the sheet means SQL NULL.
"""
import re

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, coalesce, create_map, element_at, length, lit, lower, regexp_extract, when,
)

NJ_COUNTIES = [
    "Atlantic", "Bergen", "Burlington", "Camden", "Cape May",
    "Cumberland", "Essex", "Gloucester", "Hudson", "Hunterdon",
    "Mercer", "Middlesex", "Monmouth", "Morris", "Ocean",
    "Passaic", "Salem", "Somerset", "Sussex", "Union", "Warren",
]

# Misspellings and other signals -> county (matched case-insensitively)
COUNTY_ALIASES = {
    "Altantic": "Atlantic",
    "Monmout": "Monmouth",
    "Newark": "Essex",
}


def extract_county(column_name):
    """Copy of the function under test; replace with an import from your module."""
    # One lookup of lowercase term -> county, covering counties and aliases
    lookup = {c.lower(): c for c in NJ_COUNTIES}
    lookup.update({k.lower(): v for k, v in COUNTY_ALIASES.items()})

    # Longest terms first, so a longer term beats a shorter one starting at the same spot
    county_pattern = "|".join(re.escape(t) for t in sorted(lookup, key=len, reverse=True))
    county_regex = rf"(?i).*\b({county_pattern})\b"

    # Extract the last match, then translate it to its county
    extracted = lower(regexp_extract(col(column_name), county_regex, 1))
    county_map = create_map(*[lit(x) for pair in lookup.items() for x in pair])
    return when(length(extracted) > 0, element_at(county_map, extracted)).otherwise(None)


def norm(v):
    """Sheet 'null' / NaN / '' -> None."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    v = str(v).strip()
    return None if v.lower() in ("null", "") else v


# Columns to read first: inputs, what the test produced, what was expected, pass/fail.
KEY_COLUMNS = [
    "project_name", "organization_name",
    "got_project", "got_org", "got_county",
    "expected_project", "expected_org", "expected_county",
    "project_ok", "org_ok", "county_ok", "all_ok",
]


def build_results(pdf, out, expected):
    """Full spreadsheet frame + test outputs + expected values + pass/fail flags."""
    r = pdf.reset_index(drop=True).copy()
    r["got_project"] = out["got_project"].map(norm).values
    r["got_org"] = out["got_org"].map(norm).values
    r["got_county"] = out["got_county"].map(norm).values
    r["expected_project"] = expected["project_name_county_match"].map(norm).values
    r["expected_org"] = expected["org_name_county_match"].map(norm).values
    r["expected_county"] = expected["lookup_program_county"].map(norm).values
    r["project_ok"] = [norm(g) == norm(e) for g, e in zip(r["got_project"], r["expected_project"])]
    r["org_ok"] = [norm(g) == norm(e) for g, e in zip(r["got_org"], r["expected_org"])]
    r["county_ok"] = [norm(g) == norm(e) for g, e in zip(r["got_county"], r["expected_county"])]
    r["all_ok"] = r["project_ok"] & r["org_ok"] & r["county_ok"]
    r.insert(0, "sheet_row", r.index + 2)  # row number in the spreadsheet
    rest = [c for c in r.columns if c not in KEY_COLUMNS and c != "sheet_row"]
    return r[["sheet_row"] + KEY_COLUMNS + rest]


def main(path):
    pdf = pd.read_excel(path, dtype=object)
    for c in ("project_name", "organization_name"):
        pdf[c] = pdf[c].map(norm)
    expected = pdf[["project_name_county_match", "org_name_county_match",
                    "lookup_program_county"]].map(norm)

    # build the DataFrame from the cluster's session
    spark = SparkSession.builder.getOrCreate()
    sdf = spark.createDataFrame(
        pdf[["project_name", "organization_name"]].assign(row_id=range(len(pdf))),
        "project_name string, organization_name string, row_id long",
    )
    out = (
        sdf.withColumn("got_project", extract_county("project_name"))
        .withColumn("got_org", extract_county("organization_name"))
        .withColumn("got_county", coalesce(col("got_project"), col("got_org")))
        .orderBy("row_id")
        .toPandas()
    )

    results = build_results(pdf, out, expected)

    checks = {
        "project_name match": "project_ok",
        "organization_name match": "org_ok",
        "coalesced county": "county_ok",
    }
    failed = False
    for name, ok_col in checks.items():
        n_bad = int((~results[ok_col]).sum())
        print(f"{name}: {len(results) - n_bad}/{len(results)} match")
        failed |= n_bad > 0

    # Basic invariants: no empty strings, output always Title Case.
    for c in ("got_project", "got_org", "got_county"):
        vals = results[c].dropna()
        assert (vals != "").all(), f"{c} contains empty strings"
        assert (vals == vals.str.title()).all(), f"{c} not Title Case"

    return failed, results


# Cell: configure and run
dbutils.widgets.text("spreadsheet", "/Workspace/Users/ian.davis@dca.nj.gov/claude_sandbox/tests/missing_program_county_sql_338.xlsx")
failed, results = main(dbutils.widgets.get("spreadsheet"))

In [0]:
%python
# Full results: every spreadsheet column plus the test outputs.
# project_name, organization_name, the got_* (test output) and expected_* columns come first.
display(results)

In [0]:
%python
# Same frame, with the key columns highlighted and failing rows in red.
key_in = ["project_name", "organization_name"]
key_out = ["got_project", "got_org", "got_county"]
key_exp = ["expected_project", "expected_org", "expected_county"]

def row_style(row):
    bad = not row["all_ok"]
    return ["background-color: #fdd" if bad else "" for _ in row.index]

view = results[["sheet_row"] + key_in + key_out + key_exp + ["all_ok"]]
styler = (
    view.style
    .apply(row_style, axis=1)
    .set_properties(subset=key_in, **{"font-weight": "bold"})
    .set_properties(subset=key_out, **{"background-color": "#dff0ff", "font-weight": "bold"})
    .hide(axis="index")
)
displayHTML(styler.to_html())

In [0]:
%python
# Only the rows where the test output differs from the spreadsheet.
display(results[~results["all_ok"]])
assert not failed, "extract_county test failed; see mismatches above"